# Task 5: Advanced Model Training
Cross-validation, hyperparameter tuning, and model comparison.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     GridSearchCV, StratifiedKFold)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('cardio_cleaned.csv')
X = df.drop('cardio', axis=1)
y = df['cardio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = joblib.load('scaler.pkl')
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

## 5.1 Cross-Validation (5-Fold Stratified)

In [ ]:
cv_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=11),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, model in cv_models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_results.append({
        'Model': name,
        'CV Mean': round(scores.mean(), 4),
        'CV Std': round(scores.std(), 4),
        'Folds': scores.round(4).tolist()
    })
    print(f"{name:25s} | Mean: {scores.mean():.4f} ± {scores.std():.4f} | Folds: {scores.round(4)}")

cv_df = pd.DataFrame(cv_results).sort_values('CV Mean', ascending=False).reset_index(drop=True)
cv_df[['Model', 'CV Mean', 'CV Std']]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
bars = ax.bar(cv_df['Model'], cv_df['CV Mean'], yerr=cv_df['CV Std'],
              capsize=5, color=colors, alpha=0.85, edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, cv_df['CV Mean']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('5-Fold Cross-Validation Results', fontsize=14, fontweight='bold')
ax.set_ylim(0.6, 0.85)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 5.2 Hyperparameter Tuning

In [ ]:
print("Tuning Logistic Regression...")
lr_params = {'C': [0.01, 0.1, 1, 10], 'solver': ['lbfgs', 'liblinear']}
lr_grid = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42),
                       lr_params, cv=3, scoring='accuracy', n_jobs=-1)
lr_grid.fit(X_train_scaled, y_train)
print(f"  Best: {lr_grid.best_params_} → {lr_grid.best_score_:.4f}")

print("\nTuning KNN...")
knn_params = {'n_neighbors': [5, 7, 9, 11, 15, 21], 'weights': ['uniform', 'distance']}
knn_grid = GridSearchCV(KNeighborsClassifier(), knn_params, cv=3, scoring='accuracy', n_jobs=-1)
knn_grid.fit(X_train_scaled, y_train)
print(f"  Best: {knn_grid.best_params_} → {knn_grid.best_score_:.4f}")

print("\nTuning Decision Tree...")
dt_params = {'max_depth': [5, 8, 10, 15, 20], 'min_samples_split': [2, 5, 10],
             'min_samples_leaf': [1, 2, 5]}
dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42), dt_params, cv=3,
                       scoring='accuracy', n_jobs=-1)
dt_grid.fit(X_train_scaled, y_train)
print(f"  Best: {dt_grid.best_params_} → {dt_grid.best_score_:.4f}")

print("\nTuning Random Forest...")
rf_params = {'n_estimators': [100, 200, 300], 'max_depth': [10, 15, 20],
             'min_samples_split': [2, 5]}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), rf_params,
                       cv=3, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_train_scaled, y_train)
print(f"  Best: {rf_grid.best_params_} → {rf_grid.best_score_:.4f}")

## 5.3 Save Best Tuned Models

In [ ]:
best_models = {
    'Logistic Regression': lr_grid.best_estimator_,
    'KNN': knn_grid.best_estimator_,
    'Decision Tree': dt_grid.best_estimator_,
    'Random Forest': rf_grid.best_estimator_
}

tuned_results = []
for name, model in best_models.items():
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    tuned_results.append({'Model': name, 'Tuned Accuracy': round(acc, 4)})
    joblib.dump(model, f'model_{name.lower().replace(" ", "_")}.pkl')

tuned_df = pd.DataFrame(tuned_results).sort_values('Tuned Accuracy', ascending=False)
print("Tuned Models Saved & Evaluated:")
tuned_df

In [ ]:
compare = cv_df[['Model', 'CV Mean']].merge(
    pd.DataFrame(tuned_results), on='Model', how='inner'
)
compare['Improvement'] = (compare['Tuned Accuracy'] - compare['CV Mean']).round(4)
print("Before vs After Tuning:")
compare